# TechStore — First Week Report

Initial SQL report on TechStore's products, stock levels, customer base, orders, and sales targets.  
The sample database is loaded locally into DuckDB from `SQL/seed_techstore.sql`.

In [ ]:
from pathlib import Path
import duckdb

con = duckdb.connect()
con.execute(Path("SQL/seed_techstore.sql").read_text(encoding="utf-8"))


def sql(query: str):
    """Run a SQL query against the in-memory TechStore database and return a DataFrame."""
    return con.sql(query).df()


sql("SHOW TABLES")

## 1. Product Catalog

List all products ordered by category (`categoria`) and price (`preco`, highest first). Show `nome`, `categoria`, `preco`, and `estoque`.

In [ ]:
query = """
SELECT nome, categoria, preco, estoque
FROM produtos
ORDER BY categoria ASC, preco DESC;
"""
sql(query)

**Note:** `Notebooks` has the highest-priced items (up to R$ 7,899.00 for the `Notebook Gamer Z5`), while `Acessorios` has the widest variety (8 SKUs) and the highest inventory counts (85–300 units per item).

## 2. Low Stock Items

Find products with fewer than 15 units in stock (`estoque < 15`), ordered from lowest to highest stock. Show `nome`, `categoria`, `estoque`, and `preco`.

In [ ]:
query = """
SELECT nome, categoria, estoque, preco
FROM produtos
WHERE estoque < 15
ORDER BY estoque ASC;
"""
sql(query)

**Note:** Only 3 products fall below 15 units (`Monitor UltraWide 34"` with 5, `Notebook Gamer Z5` with 8, and `Workstation W3` with 12), but all three are high-ticket items (R$ 2,599–R$ 7,899). Running out of stock on these hits revenue much harder than running low on accessories, so they should be first in line for restocking.

## 3. Customer Base by State

List all customers ordered by state (`estado`) and name (`nome`). Show `nome`, `cidade`, `estado`, and `segmento`.

In [ ]:
query = """
SELECT nome, cidade, estado, segmento
FROM clientes
ORDER BY estado ASC, nome ASC;
"""
sql(query)

**Note:** Most of the 47 customers are concentrated in the Southeast (`SP`, `RJ`, `MG`), with smaller pockets in the South (`PR`, `RS`, `SC`) and Northeast (`PE`, `BA`, `CE`), and very few accounts in the North and Center-West.

## 4. Government Segment Customers

List customers in the `governo` segment, ordered alphabetically by `nome`. Show `nome`, `cidade`, `estado`, and `segmento`.

In [ ]:
query = """
SELECT nome, cidade, estado, segmento
FROM clientes
WHERE segmento = 'governo'
ORDER BY nome ASC;
"""
sql(query)

**Note:** There are 7 government accounts spread across 6 states (`AM`, `BA`, `MG`, `PE`, `RJ`, `SP`). Because public-sector procurement usually involves tenders and longer approval cycles, tracking these separately from retail accounts makes sense for forecasting.

## 5. Top 10 Orders by Value

List the 10 largest orders by `valor_total`. Show `id`, `data_pedido`, `valor_total`, `status`, and `forma_pagamento`.

In [ ]:
query = """
SELECT id, data_pedido, valor_total, status, forma_pagamento
FROM pedidos
ORDER BY valor_total DESC
LIMIT 10;
"""
sql(query)

**Note:** The largest order is `#34` (R$ 12,450.00, paid by card, delivered). Worth flagging: two of the top 10 orders (`#91` at R$ 8,799.00 and `#155` at R$ 6,480.00) are still `pendente`, and `#41` (R$ 5,480.00) was `cancelado`.

## 6. Cancelled Orders

List all orders with `status = 'cancelado'`, ordered from highest to lowest `valor_total`. Show `id`, `data_pedido`, `valor_total`, and `forma_pagamento`.

In [ ]:
query = """
SELECT id, data_pedido, valor_total, forma_pagamento
FROM pedidos
WHERE status = 'cancelado'
ORDER BY valor_total DESC;
"""
sql(query)

**Note:** There are 12 cancelled orders totaling R$ 28,940.00. Card payments (`cartao`) account for half of them (6 orders, R$ 15,280.00), including the three largest cancellations (`#41`, `#136`, `#132`). Checking whether those are payment-gateway declines or buyer drop-offs would be a good follow-up.

## 7. Sales Reps Above R$ 50,000 Target

List sales reps with a monthly target (`meta_mensal`) above R$ 50,000, ordered from highest to lowest target. Show `nome`, `regiao`, and `meta_mensal`.

In [ ]:
query = """
SELECT nome, regiao, meta_mensal
FROM vendedores
WHERE meta_mensal > 50000
ORDER BY meta_mensal DESC;
"""
sql(query)

**Note:** Only 2 of the 7 sales reps have targets above R$ 50,000 — `Ricardo Mendes` (R$ 60,000) and `Carlos Eduardo` (R$ 55,000). Both cover `Sudeste`, which lines up with where most of the customer base is located.